User will see the output after the generated output goes into self verification layer that checks whether it is correct or not . if yes then show output . it is a safety net 

In [36]:
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

load_dotenv("../.env")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.load_local(
    "../data/faiss_index", embedding_model,
    allow_dangerous_deserialization=True
)
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
print("Ready. Chunks:", vector_store.index.ntotal)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready. Chunks: 680


In [37]:
def verify_answer(context, answer):
    check_prompt = f"""You are a strict checker.

Context:
{context}

Answer:
{answer}

Is every fact in the answer supported by the context above?
Reply with only one word: SUPPORTED or NOT_SUPPORTED"""

    result = llm.invoke(check_prompt)
    text = result.content.strip().upper()
    return "NOT_SUPPORTED" not in text and "SUPPORTED" in text


In [38]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

THRESHOLD = 0.45

def ask(question):
    results = vector_store.similarity_search_with_relevance_scores(question, k=10)
    best_score = results[0][1]
    print("Best score:", round(best_score, 2))

    if best_score < THRESHOLD:
        print("Answer: I don't have this information in the documents.")
        print("Action: Escalate to human (low confidence)")
        return

    pool = [doc for doc, score in results]
    pairs = [(question, d.page_content) for d in pool]
    r_scores = reranker.predict(pairs)
    ranked = sorted(zip(pool, r_scores), key=lambda x: x[1], reverse=True)
    docs = [d for d, s in ranked[:3]]
    print("Top chunk after rerank:", docs[0].metadata["title"])

    context = "\n\n".join(d.page_content for d in docs)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say exactly: I don't have this information in the documents.

Context:
{context}

Question: {question}

Answer:"""

    answer = llm.invoke(prompt).content

    if answer.strip().startswith("I don't have this information"):
        print("\nAnswer:", answer)
        return

    if verify_answer(context, answer):
        print("\nVerification: PASSED")
        print("\nAnswer:", answer)
    else:
        print("\nVerification: FAILED - Try one more time")
        strict_prompt = prompt + "\nBe very strict: use ONLY facts written in the context. Keep the answer short."
        answer2 = llm.invoke(strict_prompt).content

        if verify_answer(context, answer2):
            print("Verification (retry): PASSED")
            print("\nAnswer:", answer2)
            answer = answer2
        else:
            print("Verification (retry): FAILED")
            print("\nAnswer: I don't have this information in the documents.")
            print("Action: Escalate to human (answer could not be verified)")
            return

    print("\nSources:")
    seen = set()
    for d in docs:
        if d.metadata["source"] not in seen:
            seen.add(d.metadata["source"])
            print("-", d.metadata["title"], "|", d.metadata["source"])


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [39]:
ask("How do I share a page with a guest?")
ask("How do I change my Notion plan?")


Best score: 0.67
Top chunk after rerank: Sharing & permissions settings in Notion

Verification: PASSED

Answer: 1. Click **Share** at the top of the page.  
2. In the sharing dialog, type the guest’s email address (or select their name if they’re already in the workspace).  
3. Open the dropdown next to the guest’s name and choose the desired permission level.  
4. Click **Invite**.  

*Note:* The guest can be added only if the email domain is allowed for your workspace, guests are permitted by your workspace/teamspace settings, and you haven’t reached the workspace’s guest limit. If any of those conditions aren’t met, the invitation will not go through.

Sources:
- Sharing & permissions settings in Notion | https://www.notion.com/en-gb/help/sharing-and-permissions
- Build forms in Notion | https://www.notion.com/en-us/help/forms
Best score: 0.65
Top chunk after rerank: Change your Notion plan

Verification: PASSED

Answer: To change your Notion plan:

1. Open Notion on desktop or the

In [40]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

question = "How do I invite someone from outside my company to a page?"
docs = vector_store.similarity_search(question, k=10)

pairs = [(question, d.page_content) for d in docs]
scores = reranker.predict(pairs)

ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
for d, s in ranked:
    print(f"{s:.2f} | {d.metadata['title']}")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

5.74 | Sharing & permissions settings in Notion
5.11 | Intro to workspaces in Notion
3.85 | Sharing & permissions settings in Notion
2.02 | Sharing & permissions settings in Notion
1.91 | Sharing & permissions settings in Notion
-0.26 | Publish a website with Notion Sites
-0.62 | Sharing & permissions settings in Notion
-1.75 | Sharing & permissions settings in Notion
-5.62 | Sharing & permissions settings in Notion
-6.62 | Sharing & permissions settings in Notion


Implement BM25 and semantic 

In [41]:
# pip install rank_bm25

In [42]:
all_docs = list(vector_store.docstore._dict.values())
print("Total chunks:", len(all_docs))

from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(all_docs)
bm25_retriever.k = 10
print("BM25 ready")


Total chunks: 680
BM25 ready


In [ ]:
THRESHOLD = 0.45

def ask(question):
    results = vector_store.similarity_search_with_relevance_scores(question, k=10)
    best_score = results[0][1]
    print("Best score:", round(best_score, 2))

    if best_score < THRESHOLD:
        print("Answer: I don't have this information in the documents.")
        print("Action: Escalate to human (low confidence)")
        return

    queries = make_queries(question)
    print("Queries used:", len(queries))

    pool = []
    seen_texts = set()
    for q in queries:
        docs_q = vector_store.similarity_search(q, k=5) + bm25_retriever.invoke(q)[:5]
        for d in docs_q:
            if d.page_content not in seen_texts:
                seen_texts.add(d.page_content)
                pool.append(d)
    print("Pool size:", len(pool))


    pairs = [(question, d.page_content) for d in pool]
    r_scores = reranker.predict(pairs)
    ranked = sorted(zip(pool, r_scores), key=lambda x: x[1], reverse=True)
    docs = [d for d, s in ranked[:3]]
    print("Top chunk after rerank:", docs[0].metadata["title"])

    context = "\n\n".join(d.page_content for d in docs)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say exactly: I don't have this information in the documents.

Context:
{context}

Question: {question}

Answer:"""

    answer = llm.invoke(prompt).content

    if answer.strip().startswith("I don't have this information"):
        print("\nAnswer:", answer)
        return

    if verify_answer(context, answer):
        print("\nVerification: PASSED")
        print("\nAnswer:", answer)
    else:
        print("\nVerification: FAILED - Try one more time")
        strict_prompt = prompt + "\nBe very strict: use ONLY facts written in the context. Keep the answer short."
        answer2 = llm.invoke(strict_prompt).content

        if verify_answer(context, answer2):
            print("Verification (retry): PASSED")
            print("\nAnswer:", answer2)
            answer = answer2
        else:
            print("Verification (retry): FAILED")
            print("\nAnswer: I don't have this information in the documents.")
            print("Action: Escalate to human (answer could not be verified)")
            return

    print("\nSources:")
    seen = set()
    for d in docs:
        if d.metadata["source"] not in seen:
            seen.add(d.metadata["source"])
            print("-", d.metadata["title"], "|", d.metadata["source"])


Problem : if a user ask the same quesiton in different style "How do i let others see my work?" - The exact word is not present in data but the answer is present 

Solution : We can ask LLM to crete three different version of questions : any one among these three questions can match 


In [44]:
import re

def make_queries(question):
    mq_prompt =  mq_prompt = f"""Write 3 different versions of this question using different words, so a search system can find the answer in help documents.

Question: {question}
Write only the 3 questions , one per line , no numbering"""
    text = llm.invoke(mq_prompt).content
    lines= []

    for l in text.strip().split('\n'):
        l = re.sub(r"^\d+[\.\)]\s*", "", l.strip()).strip("- ").strip()
        if l :
            lines.append(l)
    return [question] + lines[:3]

def pool_for(queries):
    pool, seen = [], set()
    for q in queries:
        docs = vector_store.similarity_search(q, k=5) + bm25_retriever.invoke(q)[:5]
        for d in docs:
            if d.page_content not in seen:
                seen.add(d.page_content)
                pool.append(d)
    return pool

q = "How do I let others see my work?"
qs = make_queries(q)
for x in qs:
    print("-", x)
print("\nSingle-query pool:", len(pool_for([q])))
print("Multi-query pool:", len(pool_for(qs)))



    

- How do I let others see my work?
- How can I share my work with others?
- What steps do I need to take to make my work visible to others?
- How do I allow others to view my work?

Single-query pool: 10
Multi-query pool: 21
